In [ ]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START,END
from langchain.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

: 

In [ ]:
class Task(BaseModel):
    id:int
    title: str 

    goal: str= Field(
        ...,
        description="one sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str]=Field(
        ...,
        min_lenght=3,
        max_lenght=5,
        description="3-5 concrete, non-overlapping subpoints to cover in this section",
    )
    target_words: int= Field(
        ...,
        description="Target word count for this section (120-450).",
    )
    section_type: Literal["intro","core","examples","checklist","common_mistakes","conclusion"
                          ]=Field(
                              ...,
                              description="Use 'common_mistakes' exactly once in the plan.",
                          )
    
    brief: str= Field(...,description="What to cover")

: 

In [ ]:
class Plan(BaseModel):
    blog_title:str
    audience: str=Field(...,description="Who this blog is for.")
    tone:str=Field(...,description="writing tone(e.g., practical,crisp)")
    tasks: List[Task]

: 

In [ ]:
class State(TypedDict):
    topic: str
    plan: Plan
    #reducer: results from workers get concatenated automatically
    sections: Annotated[list[str],operator.add]
    final:str

: 

In [ ]:
llm=ChatOpenAI(model="gpt-4.1-mini")


: 

In [ ]:
def orchestrator(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke(
        [
            SystemMessage{
                content=("Create a blog plan with 5-7 sections on the following topic.")
            },
            HumanMessage{content=f"Topic:{state['topic']}"},
        ]  
    )
    return {"plan":plan}

: 

In [ ]:
#fanout to check how mnay tasks are their in plan object
def fanout(state:State):
    return [Send("worker",{"task":task,"topic":state["topic"],"plan":state["plan"]})
            for task in state["plan"].tasks]


: 

In [ ]:
def worker(payload:dict)->dict:

    #payload contains what we sent
    task=payload["task"]
    topic=payload["topic"]
    plan=payload["plan"]

    blog_title=plan.blog_title

    section_md=llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section"),
            HumanMessage(
                content=(
                    f"Blog:{blog_title}\n"
                    f"Audience :{plan.audience}\n"
                    f"Tone:{plan.tone}"
                    f"Topic:{topic}\n"
                    f"Section:{task.title}\n"
                    f"Section Type:{task.section_type}\n"
                    f"Goal:{task.goal}\n"
                    f"Target Words:{task.target_words}\n"
                    f"Bulets: {task.bullets}\n\n"
                    "Return only the section content in markdown."
                )
            )
        ]
    ).content.strip()

    return {"sections":[section_md]}




: 

In [ ]:
from pathlib import Path
def reducer(state:State)->dict:
    title=state["plan"].blog_title
    body="\n\n".join(state["sections"]).strip()

    final_md=f"# {title}\n\n{body}\n"

    #to save file
    filename=title.lower().replace(" ","_")+".md"
    output_path=Path(filename)
    output_path.write_text(final_md,encodign="utf-8")

    return {"final":final_md}


: 

In [ ]:
g=StateGraph(State)
g.add_node("orchestrator",orchestrator)
g.add_node("worker",worker)
g.add_node("reducer",reducer)


: 

In [ ]:
g.add_edge(START,"orchestrator")
g.add_conditional_edges("orchestrator",fanout,["worker"])
g.add_edge("worker","reducer")
g.add_edge(worker,END)

app=g.compile()

app

: 

In [ ]:
out=app.invoke({"topic":"Write a blog on self-attention"})

: 